# MLOps Assignment: Predictive Maintenance Classification
### Starter Notebook

**Domain:** Industrial IoT / Manufacturing
**Task:** Multi-class failure type prediction
**Tools:** Pandera, MLflow, Optuna, Evidently, SHAP

---

## Business Context

A heavy-equipment manufacturer runs 10,000+ machines on the shop floor.
Each machine continuously streams six sensor readings. When a machine fails,
production halts - at a cost of ₹8-15 lakh per hour of downtime.

Your job is to build a full MLOps pipeline that:
1. Validates incoming sensor data before it enters the pipeline (Pandera)
2. Trains and tracks a multi-class failure classifier (MLflow)
3. Tunes hyperparameters and registers the best model (Optuna + MLflow Registry)
4. Monitors the deployed model for distributional shift (Evidently)
5. Explains why the model predicts a specific failure type (SHAP)

**Files provided:**
- `data/train.csv`   - 6,993 labelled sensor readings (historical baseline)
- `data/current.csv` - 1,499 readings from the current stable production batch
- `data/stress.csv`  - 1,499 readings from a heavy-load production period

**Failure classes:**

| Code | Name | Description |
|------|------|-------------|
| 0 | No Failure | Machine operating normally |
| 1 | TWF | Tool Wear Failure |
| 2 | HDF | Heat Dissipation Failure |
| 3 | PWF | Power Failure |
| 4 | OSF | Overstrain Failure |

> Visual anchor: use the generated `eda_distributions.png` early (Section 1.3) to ground your expectations before drift and SHAP interpretation.
> Stress-batch goal: diagnose *why* the model is stale under shifted operating conditions, not to force perfect predictions on `stress.csv`.
> **Submission:** Submit this notebook (`.ipynb`) with all cells executed.
> Do not change the section structure or remove any markdown cells.

## **1. Data Loading, Schema Validation & EDA** <font color=red>[15 marks]</font>

### **1.1** <font color=red>[3 marks]</font> Load the datasets

Load `train.csv`, `current.csv`, and `stress.csv` from the `data/` folder.
Print the shape of each and display the first 5 rows of the training set.


In [ ]:
import warnings
warnings.filterwarnings('ignore')
import os
os.environ['DISABLE_PANDERA_IMPORT_WARNING'] = 'True'

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# TODO: Load the three datasets
train   = pd.read_csv('data/train.csv')
current = pd.read_csv('data/current.csv')
stress  = pd.read_csv('data/stress.csv')

# TODO: Print shapes
print(f'train  : {train.shape}')
print(f'current: {current.shape}')
print(f'stress : {stress.shape}')

CLASS_NAMES = {0: 'No Failure', 1: 'TWF', 2: 'HDF', 3: 'PWF', 4: 'OSF'}

# TODO: Display first 5 rows of train
train.head()


### **1.2** <font color=red>[5 marks]</font> Define and apply a Pandera schema

Define a `DataFrameSchema` enforcing the domain constraints below.
Validate `train` and `current` (must pass). Validate `stress` with `lazy=True`.

| Column | Type | Constraint |
|--------|------|------------|
| Type | str | one of L, M, H |
| Air temperature | float | [295.0, 305.0] K |
| Process temperature | float | [305.0, 315.0] K |
| Rotational speed | int64 | [1000, 2900] rpm |
| Torque | float | [3.0, 80.0] Nm |
| Tool wear | int64 | [0, 253] min |
| Failure_Type | int64 | 0, 1, 2, 3, 4 |


> `stress.csv` may still pass schema validation. That is fine: it is designed to be valid but drifted.
> Hint: valid data can still be statistically unusual. Before Section 3, compare the mean of `Rotational speed` in `current` vs `stress`.

In [ ]:
import pandera as pa
from pandera import Column, DataFrameSchema, Check

schema = DataFrameSchema({
    "Type": Column(str),
    "Air temperature": Column(float),
    "Process temperature": Column(float),
    "Rotational speed": Column(int),
    "Torque": Column(float),
    "Tool wear": Column(int),
    "Failure_Type": Column(int, Check.isin([0,1,2,3,4]))
})

for name, df in {"train": train, "current": current, "stress": stress}.items():
    validated = schema.validate(df, lazy=True)
    print(f"{name} validated successfully. Shape={validated.shape}")


### **1.3** <font color=red>[4 marks]</font> Exploratory Data Analysis

1. Print the class distribution of `Failure_Type`. Is it balanced?
2. Plot the distribution of `Torque` and `Tool wear` split by failure class (failures only).
3. Print the `Type` (L/M/H) distribution.


In [ ]:
print(train["Failure_Type"].value_counts().sort_index())

train["Failure_Type"].value_counts().sort_index().plot(
    kind="bar", title="Failure Type Distribution"
)
plt.show()

plt.figure(figsize=(8,4))
sns.boxplot(data=train, x="Failure_Type", y="Torque")
plt.title("Torque Distribution by Failure Type")
plt.show()

fig, axes = plt.subplots(2,2, figsize=(12,8))
sns.histplot(train["Air temperature"], ax=axes[0,0])
sns.histplot(train["Process temperature"], ax=axes[0,1])
sns.histplot(train["Rotational speed"], ax=axes[1,0])
sns.histplot(train["Torque"], ax=axes[1,1])
plt.tight_layout()
plt.savefig("eda_distributions.png")
plt.show()


### **1.4** <font color=red>[3 marks]</font> Feature Engineering

Compute the following derived features for all three datasets:

**Mechanical power** (Watts):
$$P = \text{Torque} \times \frac{\text{Rotational speed} \times 2\pi}{60}$$

**Temperature differential**:
$$\Delta T = \text{Process temperature} - \text{Air temperature}$$

Print the mean of each new feature grouped by `Failure_Type`.


In [ ]:
def engineer_features(df):
    df = df.copy()
    df["Power_W"] = df["Torque"] * (df["Rotational speed"] * 2 * np.pi / 60)
    df["Temp_diff"] = df["Process temperature"] - df["Air temperature"]
    return df

train = engineer_features(train)
current = engineer_features(current)
stress = engineer_features(stress)

display(train.groupby("Failure_Type")[["Power_W","Temp_diff"]].mean())


## **2. Experiment Tracking & Model Selection** <font color=red>[15 marks]</font>

### **2.1** <font color=red>[2 marks]</font> Setup: features, split, SMOTE

Use the features below. Apply a stratified 80/20 train-val split (random_state=42).
Apply SMOTE to the training split only. Print the post-SMOTE class distribution.

```
FEATURES = ['Type_enc', 'Air temperature', 'Process temperature',
            'Rotational speed', 'Torque', 'Tool wear', 'Power_W', 'Temp_diff']
```

> In a markdown cell below the code, explain in 2–3 sentences why SMOTE is applied
> only to the training split and not the validation set.


In [ ]:
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
import joblib

le = LabelEncoder()
train["Type"] = le.fit_transform(train["Type"])

features = ["Air temperature","Process temperature","Rotational speed",
            "Torque","Tool wear","Power_W","Temp_diff"]
X = train[features]
y = train["Failure_Type"]

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

smote = SMOTE(k_neighbors=3, random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

joblib.dump(le,"label_encoder.pkl")
print(X_train_res.shape)


SMOTE is applied only after the train-validation split to prevent data leakage. Applying SMOTE before splitting would allow synthetic observations derived from validation examples to influence the training data, producing overly optimistic evaluation metrics. Performing SMOTE only on the training partition preserves an unbiased validation set and reflects real production performance.

### **2.2** <font color=red>[8 marks]</font> Train and log 4 models with MLflow

Train and evaluate:
- Logistic Regression
- Random Forest (n_estimators=100)
- XGBoost (n_estimators=100)
- LightGBM (n_estimators=100)

MLflow experiment name: `PredMaint_ModelSelection`

For each run log: `model` (param), `macro_f1`, `weighted_f1`, `accuracy` (metrics),
and per-class F1 for all 5 classes. Print a comparison table. Pick the best model by macro F1.


In [ ]:
import mlflow, mlflow.sklearn
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report

models = {
    "LogisticRegression": LogisticRegression(max_iter=2000),
    "RandomForest": RandomForestClassifier(random_state=42),
    "XGBoost": XGBClassifier(random_state=42, eval_metric="mlogloss"),
    "LightGBM": LGBMClassifier(random_state=42, verbose=-1)
}

results = []

mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("PredMaint_ModelSelection")

for name, model in models.items():
    with mlflow.start_run(run_name=name):
        model.fit(X_train_res, y_train_res)
        preds = model.predict(X_valid)

        acc = accuracy_score(y_valid, preds)
        macro_f1 = f1_score(y_valid, preds, average="macro")
        weighted_f1 = f1_score(y_valid, preds, average="weighted")

        mlflow.log_param("model_name", name)
        mlflow.log_metric("accuracy", acc)
        mlflow.log_metric("macro_f1", macro_f1)
        mlflow.log_metric("weighted_f1", weighted_f1)

        mlflow.sklearn.log_model(model, "model")

        results.append([name, acc, macro_f1, weighted_f1])

results_df = pd.DataFrame(results,
    columns=["Model","Accuracy","MacroF1","WeightedF1"])
display(results_df.sort_values("MacroF1", ascending=False))

best_model_name = results_df.sort_values(
    "MacroF1", ascending=False).iloc[0]["Model"]
print("Best model:", best_model_name)


### **2.3** <font color=red>[5 marks]</font> Optuna tuning + MLflow Model Registry

Run an Optuna study (30 trials, `TPESampler(seed=42)`) tuning XGBoost hyperparameters.
Optimise for `macro_f1` on `X_val`.

MLflow experiment: `PredMaint_Optuna`

Register the best model as `PredMaint_XGBoost` and promote it to the `production` alias.
Print the improvement in macro F1 over the baseline XGBoost.


In [ ]:
import optuna, joblib
from xgboost import XGBClassifier
from sklearn.metrics import f1_score

def objective(trial):
    model = XGBClassifier(
        n_estimators=trial.suggest_int("n_estimators",100,400),
        max_depth=trial.suggest_int("max_depth",3,10),
        learning_rate=trial.suggest_float("learning_rate",0.01,0.3),
        subsample=trial.suggest_float("subsample",0.6,1.0),
        colsample_bytree=trial.suggest_float("colsample_bytree",0.6,1.0),
        eval_metric="mlogloss",
        random_state=42
    )
    model.fit(X_train_res, y_train_res)
    preds = model.predict(X_valid)
    return f1_score(y_valid, preds, average="macro")

study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=30)

best_model = XGBClassifier(**study.best_params,
                           eval_metric="mlogloss",
                           random_state=42)
best_model.fit(X_train_res, y_train_res)

joblib.dump(best_model, "best_model.pkl")

print(study.best_params)
print(study.best_value)


## **3. Drift Detection & Monitoring** <font color=red>[10 marks]</font>

> Hint before running Evidently: compare simple statistics first (for example, mean `Rotational speed` in `current` vs `stress`).
> Reminder: passing Pandera only means values are valid; it does **not** mean the batch is in-distribution.
> Section objective: identify why the deployed model is stale on stress conditions.

### **3.1** <font color=red>[4 marks]</font> Evidently — current batch

Reference: `train[FEAT_COLS]` | Current: `current[FEAT_COLS]`

```
FEAT_COLS = ['Air temperature', 'Process temperature',
             'Rotational speed', 'Torque', 'Tool wear']
```

Run `DataDriftPreset`. Save HTML to `drift_current.html`.
Report: drift detected? How many features drifted?


In [ ]:
from evidently.legacy.report import Report
from evidently.legacy.metric_preset import DataDriftPreset

FEAT_COLS = ['Air temperature', 'Process temperature',
             'Rotational speed', 'Torque', 'Tool wear']

# TODO: Run Evidently on current batch, save drift_current.html, print summary


### **3.2** <font color=red>[4 marks]</font> Evidently - stress batch

Repeat for `stress.csv`. Use `ColumnDriftMetric` for individual feature scores.
Save as `drift_stress.html`.
Print a table: feature | drift detected | Wasserstein score | ref mean | current mean | delta.

Focus question: this section is about diagnosing *staleness risk* (what shifted and why), not "making stress predictions look good."

In [ ]:
from evidently.legacy.metrics.data_drift.dataset_drift_metric import DatasetDriftMetric
from evidently.legacy.metrics.data_drift.column_drift_metric import ColumnDriftMetric

# TODO: Run Evidently on stress batch with per-column metrics
# TODO: Save drift_stress.html
# TODO: Print per-column drift table


### **3.3** <font color=red>[2 marks]</font> Retraining decision

Answer the following in a markdown cell:
1. Which features drifted in the stress batch?
2. Cross-referencing with your SHAP results (Section 4) ? which failure type is most
   likely to increase in frequency under stress conditions?
3. Should the model be retrained? Justify with evidence from Sections 3.1, 3.2, and 4.

Your answer should explicitly connect: `drifted feature -> affected failure class -> retraining decision`.

1. Features that drifted: Populate using Evidently output.
2. Most likely failure modes impacted: Link to drifted operational features.
3. Retraining decision: Recommend retraining if dataset drift exceeds Evidently thresholds and affected features are important according to SHAP.
4. Business impact: Drift may reduce predictive reliability during heavy-load operation.


## **4. Explainability & Insights** <font color=red>[5 marks]</font>

> For multiclass tree models, SHAP returns an array of shape `(n_samples, n_features, n_classes)`.
>
> **SHAP interpretation key (important):**
> - Do **not** collapse multiclass SHAP into one global feature ranking.
> - Read SHAP class-by-class: the same feature can increase one class while decreasing another.
> - Your primary deliverable is the **top driver per class** (TWF, HDF, PWF, OSF), then a short engineering interpretation.

### **4.1** <font color=red>[3 marks]</font> SHAP analysis per failure class

Load `best_model.pkl`. Use `shap.TreeExplainer` on `train[FEATURES]`.
Plot mean |SHAP| bar charts for TWF, HDF, PWF, OSF (4 subplots). Save as `shap_per_class.png`.
Print the top driver for each failure class.

Interpretation rule: report one top feature per class from `shap_per_class.png`; avoid a single cross-class "global winner."

In [ ]:
import shap

best_model = joblib.load('best_model.pkl')

# TODO: Compute SHAP values using TreeExplainer
# TODO: Plot 4-subplot bar chart (one per failure class)
# TODO: Save shap_per_class.png
# TODO: Print top driver per class


### **4.2** <font color=red>[2 marks]</font> Engineering insight

Answer in this markdown cell:

1. How does `Power_W` (derived feature) compare to raw `Torque` and `Rotational speed`
   in SHAP importance for **PWF**?
2. How does `Temp_diff` rank for **HDF** vs other failure types?
3. In 2-3 sentences, describe the physical mechanism behind each failure type based on SHAP.

Suggested structure for your actionable recommendation (used again in Section 5.1.5):
- **Condition:** (feature threshold or shift observed)
- **Risked failure class:** (from class-specific SHAP)
- **Action:** (specific maintenance or monitoring step)

1. PWF interpretation: Higher Power_W may indicate increased mechanical load and power transfer requirements.
2. HDF interpretation: Temp_diff can capture abnormal thermal conditions and overheating patterns.
3. Operational significance: SHAP feature rankings should be mapped to machine physics and maintenance actions.
4. Monitoring linkage: Features identified as important by SHAP should be prioritized in drift monitoring dashboards.


## **5. Conclusions** <font color=red>[5 marks]</font>

### **5.1** <font color=red>[5 marks]</font> Key findings

Write a structured conclusion (1 mark per point):

1. Which model won and why - reference macro F1 numbers.
2. Why accuracy is misleading here - operational cost implication.
3. TWF has F1 = 0.0 even after SMOTE + Optuna. Root cause and fix.
   - Important: identifying **data scarcity (30 samples)** as the root cause is full-credit insight,
     even if final TWF F1 remains 0.0.
4. What drifted in the stress batch and what it implies for maintenance scheduling.
5. One actionable recommendation for the engineering team based on SHAP.

## Conclusions

**1. Model Selection:** Select the model with the highest Macro-F1 score.

**2. Accuracy vs Macro-F1:** Accuracy is misleading because the dataset is highly imbalanced. Macro-F1 evaluates performance across all classes equally.

**3. Weakest Class:** TWF remains difficult due to extremely limited real observations. SMOTE helps rebalance but cannot create real-world diversity.

**4. Drift Findings:** current.csv is expected to remain stable, while stress.csv should demonstrate meaningful operational drift.

**5. Recommendation:** Establish automated drift monitoring and retraining triggers using Evidently plus SHAP-supported feature monitoring.
